# Predicting 2026 World Cup Winner (held in Canada, USA , Mexico)
## 2026 World Cup is different; The tournament expands from 32 teams to 48 teams.
## <span style="color:green">Aim - "1st, 2nd, 3rd and 4th place countries?"</span>	
#### worldcup.json file with historical matches data
#### Entities to be used : matches, tournament_standings, host_countries

In [1]:
# reading worldcup.json file
import json
import pandas as pd

with open('./worldcup.json', 'r') as file:
    data = json.load(file)
    
data.keys()

dict_keys(['tournaments', 'confederations', 'teams', 'players', 'managers', 'referees', 'stadiums', 'matches', 'awards', 'qualified_teams', 'squads', 'manager_appointments', 'referee_appointments', 'team_appearances', 'player_appearances', 'manager_appearances', 'referee_appearances', 'goals', 'penalty_kicks', 'bookings', 'substitutions', 'host_countries', 'tournament_stages', 'groups', 'group_standings', 'tournament_standings', 'award_winners'])

In [2]:
pd.set_option('display.max_columns', None)

In [3]:
#*** this is to view values of each key
# for key in data.keys():
#     print(f"\n---{key.upper()}")
#     df = pd.DataFrame(data[key])
#     print(df.head(2))

### All historical mens world cup data

In [4]:
# Extract only the necessary entities from worldcup.json file
# All entities contain only mens world cup data
df_appearances = pd.DataFrame(data['team_appearances']) # tournament_name -> only mens world cup
df_appearances = df_appearances[~df_appearances['tournament_name'].str.contains("women", case=False)]

df_matches = pd.DataFrame(data['matches']) # tournament_name -> only mens world cup
df_matches = df_matches[~df_matches['tournament_name'].str.contains("women", case=False)]

df_standings = pd.DataFrame(data['tournament_standings']) # tournament_name -> only mens world cup
df_standings = df_standings[~df_standings['tournament_name'].str.contains("women", case=False)]

df_hosts = pd.DataFrame(data['host_countries']) # tournament_name -> only mens world cup
df_hosts = df_hosts[~df_hosts['tournament_name'].str.contains("women", case=False)]

### Mens world cup data since 1998
#### when the tournament expanded to 32 teams (high quality, modern)

In [5]:
# All entities contain only mens world cup data since 1998
# tournament_id -> since 1998
df_appearances_1998 = df_appearances[df_appearances['tournament_id'].str[-4:].astype(int) >= 1998]

df_matches_1998 = df_matches[df_matches['tournament_id'].str[-4:].astype(int) >= 1998]

df_standings_1998 = df_standings[df_standings['tournament_id'].str[-4:].astype(int) >= 1998]

df_hosts_1998 = df_hosts[df_hosts['tournament_id'].str[-4:].astype(int) >= 1998]

In [6]:
df_matches_1998.head()

,key_id,tournament_id,tournament_name,match_id,match_name,stage_name,group_name,group_stage,knockout_stage,replayed,replay,match_date,match_time,stadium_id,stadium_name,city_name,country_name,home_team_id,home_team_name,home_team_code,away_team_id,away_team_name,away_team_code,score,home_team_score,away_team_score,home_team_score_margin,away_team_score_margin,extra_time,penalty_shootout,score_penalties,home_team_score_penalties,away_team_score_penalties,result,home_team_win,away_team_win,draw
568,569,WC-1998,1998 FIFA Men's World Cup,M-1998-01,Brazil vs Scotland,group stage,Group A,1,0,0,0,1998-06-10,17:30,S-073,Stade de France,Saint-Denis,France,T-09,Brazil,BRA,T-64,Scotland,SCO,2–1,2,1,1,-1,0,0,0-0,0,0,home team win,1,0,0
569,570,WC-1998,1998 FIFA Men's World Cup,M-1998-02,Morocco vs Norway,group stage,Group A,1,0,0,0,1998-06-10,21:00,S-065,Stade de la Mosson,Montpellier,France,T-47,Morocco,MAR,T-53,Norway,NOR,2–2,2,2,0,0,0,0,0-0,0,0,draw,0,0,1
570,571,WC-1998,1998 FIFA Men's World Cup,M-1998-03,Italy vs Chile,group stage,Group B,1,0,0,0,1998-06-11,17:30,S-056,Stade du Parc Lescure,Bordeaux,France,T-41,Italy,ITA,T-13,Chile,CHL,2–2,2,2,0,0,0,0,0-0,0,0,draw,0,0,1
571,572,WC-1998,1998 FIFA Men's World Cup,M-1998-04,Cameroon vs Austria,group stage,Group B,1,0,0,0,1998-06-11,21:00,S-076,Stade de Toulouse,Toulouse,France,T-11,Cameroon,CMR,T-05,Austria,AUT,1–1,1,1,0,0,0,0,0-0,0,0,draw,0,0,1
572,573,WC-1998,1998 FIFA Men's World Cup,M-1998-05,Paraguay vs Bulgaria,group stage,Group D,1,0,0,0,1998-06-12,14:30,S-065,Stade de la Mosson,Montpellier,France,T-55,Paraguay,PRY,T-10,Bulgaria,BGR,0–0,0,0,0,0,0,0,0-0,0,0,draw,0,0,1


### feature engineering

In [20]:
# calculate team stats, goals made and goals conceded
team_stats = df_appearances_1998.groupby('team_code').agg({
    'goals_for' : 'mean',
    'goals_against' : 'mean',
    'win' : 'mean'
}).reset_index()

# merge data frames
merged_df = df_matches_1998.merge(team_stats, left_on='home_team_code', right_on='team_code')
merged_df = merged_df.merge(team_stats, left_on='away_team_code', right_on='team_code', suffixes=('_home', '_away'))

# Calculate the difference in strength
# Calculate Net Rating (Offense - Defense) for both sides
merged_df['home_net_rating'] = merged_df['goals_for_home'] - merged_df['goals_against_home']
merged_df['away_net_rating'] = merged_df['goals_for_away'] - merged_df['goals_against_away']

# Create the final "Gap" feature for the model
# This now considers both how much they score AND how much they concede
merged_df['goal_diff_exp'] = merged_df['home_net_rating'] - merged_df['away_net_rating']

# Calculate the win rate difference
merged_df['win_rate_diff'] = merged_df['win_home'] - merged_df['win_away']

# Define your Features (X) and your Target (y)
# We want the model to look at the gaps to predict if the Home Team wins
X = merged_df[['goal_diff_exp', 'win_rate_diff']]
y = merged_df['home_team_win'] # 1 if Home won, 0 if Away won/Draw

# Split the data (80% for learning, 20% for testing)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and Train the Model
from sklearn.ensemble import RandomForestClassifier

# Create the model
model = RandomForestClassifier(random_state=42)

# Train the model using the historical data
model.fit(X_train, y_train)

# Print a success message
print("Model trained successfully!")

# Print the accuracy of the trained model
accuracy = model.score(X_test, y_test)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

Model trained successfully!
Model Accuracy: 68.89%


In [8]:
merged_df.head()

,key_id,tournament_id,tournament_name,match_id,match_name,stage_name,group_name,group_stage,knockout_stage,replayed,replay,match_date,match_time,stadium_id,stadium_name,city_name,country_name,home_team_id,home_team_name,home_team_code,away_team_id,away_team_name,away_team_code,score,home_team_score,away_team_score,home_team_score_margin,away_team_score_margin,extra_time,penalty_shootout,score_penalties,home_team_score_penalties,away_team_score_penalties,result,home_team_win,away_team_win,draw,team_code_home,goals_for_home,goals_against_home,win_home,team_code_away,goals_for_away,goals_against_away,win_away,home_net_rating,away_net_rating,goal_diff_exp,win_rate_diff
0,569,WC-1998,1998 FIFA Men's World Cup,M-1998-01,Brazil vs Scotland,group stage,Group A,1,0,0,0,1998-06-10,17:30,S-073,Stade de France,Saint-Denis,France,T-09,Brazil,BRA,T-64,Scotland,SCO,2–1,2,1,1,-1,0,0,0-0,0,0,home team win,1,0,0,BRA,1.902439,0.975610,0.707317,SCO,0.666667,2.000000,0.000000,0.926829,-1.333333,2.260163,0.707317
1,570,WC-1998,1998 FIFA Men's World Cup,M-1998-02,Morocco vs Norway,group stage,Group A,1,0,0,0,1998-06-10,21:00,S-065,Stade de la Mosson,Montpellier,France,T-47,Morocco,MAR,T-53,Norway,NOR,2–2,2,2,0,0,0,0,0-0,0,0,draw,0,0,1,MAR,1.000000,1.076923,0.384615,NOR,1.250000,1.250000,0.250000,-0.076923,0.000000,-0.076923,0.134615
2,571,WC-1998,1998 FIFA Men's World Cup,M-1998-03,Italy vs Chile,group stage,Group B,1,0,0,0,1998-06-11,17:30,S-056,Stade du Parc Lescure,Bordeaux,France,T-41,Italy,ITA,T-13,Chile,CHL,2–2,2,2,0,0,0,0,0-0,0,0,draw,0,0,1,ITA,1.409091,0.818182,0.500000,CHL,1.166667,1.416667,0.333333,0.590909,-0.250000,0.840909,0.166667
3,572,WC-1998,1998 FIFA Men's World Cup,M-1998-04,Cameroon vs Austria,group stage,Group B,1,0,0,0,1998-06-11,21:00,S-076,Stade de Toulouse,Toulouse,France,T-11,Cameroon,CMR,T-05,Austria,AUT,1–1,1,1,0,0,0,0,0-0,0,0,draw,0,0,1,CMR,0.733333,1.733333,0.133333,AUT,1.000000,1.333333,0.000000,-1.000000,-0.333333,-0.666667,0.133333
4,573,WC-1998,1998 FIFA Men's World Cup,M-1998-05,Paraguay vs Bulgaria,group stage,Group D,1,0,0,0,1998-06-12,14:30,S-065,Stade de la Mosson,Montpellier,France,T-55,Paraguay,PRY,T-10,Bulgaria,BGR,0–0,0,0,0,0,0,0,0-0,0,0,draw,0,0,1,PRY,0.875000,0.812500,0.312500,BGR,0.333333,2.333333,0.000000,0.062500,-2.000000,2.062500,0.312500


In [23]:
import pandas as pd

# 1. Filter your team_stats to ONLY include the 48 participants
current_participants_stats = team_stats[team_stats['team_code'].isin(participants)].copy()

# 2. Calculate the "Power Level" of each team (Net Rating)
current_participants_stats['net_rating'] = (
    current_participants_stats['goals_for'] - current_participants_stats['goals_against']
)

# 3. Calculate the "World Average" based on these 48 teams
avg_net = current_participants_stats['net_rating'].mean()
avg_win = current_participants_stats['win'].mean()

power_rankings = []

for team in participants:
    stats = current_participants_stats[current_participants_stats['team_code'] == team]
    
    if not stats.empty:
        team_net = stats['net_rating'].values[0]
        team_win = stats['win'].values[0]
        
        # CREATE THE GAPS: This is what the model understands
        goal_gap = team_net - avg_net
        win_gap = team_win - avg_win
        
        # Format for the model
        input_data = pd.DataFrame([[goal_gap, win_gap]], 
                                 columns=['goal_diff_exp', 'win_rate_diff'])
        
        # Predict probability of winning against the "Average Participant"
        prob = model.predict_proba(input_data)[0][1] 
        
        power_rankings.append({'Team': team, 'Rating': prob})

# 4. Display the results
df_final = pd.DataFrame(power_rankings).sort_values(by='Rating', ascending=False)
print(df_final)

   Team    Rating
16  BRA  0.910000
32  SWE  0.855000
17  COL  0.819167
28  FRA  0.785000
18  ECU  0.690000
15  ARG  0.685000
25  HRV  0.670000
11  MAR  0.655000
10  GHA  0.625000
26  CZE  0.625000
34  TUR  0.625000
1   MEX  0.612500
12  SEN  0.594000
33  CHE  0.593333
20  URY  0.579167
9   EGY  0.559167
30  SCO  0.559167
8   CIV  0.429000
24  BIH  0.390000
23  BEL  0.339167
19  PRY  0.263500
22  AUT  0.239833
4   IRN  0.210000
3   AUS  0.210000
7   QAT  0.205833
27  ENG  0.198667
31  ESP  0.170000
5   JPN  0.160000
21  NZL  0.146500
29  NOR  0.133333
0   CAN  0.113333
14  PAN  0.082500
6   KOR  0.080000
2   USA  0.080000
13  TUN  0.000000


In [26]:
print(team_stats[team_stats['team_code'].isin(['ARG', 'TUR', 'BRA', 'COL','SWE','HRV'])])

   team_code  goals_for  goals_against       win
1        ARG   1.722222       1.000000  0.694444
7        BRA   1.902439       0.975610  0.707317
14       COL   1.583333       0.833333  0.583333
28       HRV   1.433333       1.100000  0.566667
58       SWE   1.076923       1.000000  0.384615
62       TUR   1.428571       0.857143  0.571429
